# 如何使用 tiktoken 计算 token 数

[`tiktoken`](https://github.com/openai/tiktoken/blob/main/README.md) 是 OpenAI 推出的一款快速开源分词器。

给定一个文本字符串（例如 `"tiktoken is great!"`）和一种编码（例如 `"cl100k_base"`），分词器可以将文本字符串拆分为一个 token 列表（例如 `["t", "ik", "token", " is", " great", "!"]`）。

将文本字符串拆分为 token 很有用，因为 GPT 模型是以 token 的形式看待文本的。知道一个文本字符串包含多少 token，可以告诉你 (a) 该字符串对于文本模型来说是否过长，以及 (b) 一次 OpenAI API 调用的费用是多少（因为用量按 token 计价）。


## 编码

编码规定了文本如何转换为 token。不同的模型使用不同的编码。

`tiktoken` 支持 OpenAI 模型使用的三种编码：

| 编码名称           | OpenAI 模型                                       |
|-------------------------|-----------------------------------------------------|
| `o200k_base`            | `gpt-4o`、`gpt-4o-mini`                             |
| `cl100k_base`           | `gpt-4-turbo`、`gpt-4`、`gpt-3.5-turbo`、`text-embedding-ada-002`、`text-embedding-3-small`、`text-embedding-3-large`  |
| `p50k_base`             | Codex 模型、`text-davinci-002`、`text-davinci-003`|
| `r50k_base`（或 `gpt2`） | GPT-3 模型，如 `davinci`                         |

你可以使用 `tiktoken.encoding_for_model()` 获取某个模型的编码，如下所示：
```python
encoding = tiktoken.encoding_for_model('gpt-4o-mini')
```

注意，`p50k_base` 与 `r50k_base` 有大量重叠，对于非代码类应用，它们通常会给出相同的 token。

## 各语言的分词器库

对于 `o200k_base`、`cl100k_base` 和 `p50k_base` 编码：
- Python：[tiktoken](https://github.com/openai/tiktoken/blob/main/README.md)
- .NET / C#：[SharpToken](https://github.com/dmitry-brazhenko/SharpToken)、[TiktokenSharp](https://github.com/aiqinxuancai/TiktokenSharp)
- Java：[jtokkit](https://github.com/knuddelsgmbh/jtokkit)
- Golang：[tiktoken-go](https://github.com/pkoukk/tiktoken-go)
- Rust：[tiktoken-rs](https://github.com/zurawiki/tiktoken-rs)

对于 `r50k_base`（`gpt2`）编码，许多语言都有可用的分词器。
- Python：[tiktoken](https://github.com/openai/tiktoken/blob/main/README.md)（或者 [GPT2TokenizerFast](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2TokenizerFast)）
- JavaScript：[gpt-3-encoder](https://www.npmjs.com/package/gpt-3-encoder)
- .NET / C#：[GPT Tokenizer](https://github.com/dluc/openai-tools)
- Java：[gpt2-tokenizer-java](https://github.com/hyunwoongko/gpt2-tokenizer-java)
- PHP：[GPT-3-Encoder-PHP](https://github.com/CodeRevolutionPlugins/GPT-3-Encoder-PHP)
- Golang：[tiktoken-go](https://github.com/pkoukk/tiktoken-go)
- Rust：[tiktoken-rs](https://github.com/zurawiki/tiktoken-rs)

（OpenAI 不对第三方库作任何背书或保证。）


## 字符串通常如何被分词

在英语中，token 的长度通常从一个字符到一个单词不等（例如 `"t"` 或 `" great"`），但在某些语言中，token 可能短于一个字符，也可能长于一个单词。空格通常与单词开头归为一组（例如 `" is"`，而不是 `"is "` 或 `" "`+`"is"`）。你可以在 [OpenAI Tokenizer](https://beta.openai.com/tokenizer) 或第三方的 [Tiktokenizer](https://tiktokenizer.vercel.app/) 网页应用中快速查看字符串是如何被分词的。

## 0. 安装 `tiktoken`

如有需要，使用 `pip` 安装 `tiktoken`：

In [1]:
%pip install --upgrade tiktoken -q
%pip install --upgrade openai -q


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. 导入 `tiktoken`

In [2]:
import tiktoken

## 2. 加载编码

使用 `tiktoken.get_encoding()` 按名称加载编码。

首次运行时需要联网下载，之后运行则无需联网。

In [3]:
encoding = tiktoken.get_encoding("cl100k_base")


使用 `tiktoken.encoding_for_model()` 根据给定的模型名称自动加载正确的编码。

In [4]:
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

## 3. 使用 `encoding.encode()` 将文本转换为 token



`.encode()` 方法将文本字符串转换为 token 整数列表。

In [5]:
encoding.encode("tiktoken is great!")

[83, 8251, 2488, 382, 2212, 0]

通过计算 `.encode()` 返回列表的长度来统计 token 数。

In [6]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [7]:
num_tokens_from_string("tiktoken is great!", "o200k_base")

6

## 4. 使用 `encoding.decode()` 将 token 转换为文本

`.decode()` 将 token 整数列表转换为字符串。

In [8]:
encoding.decode([83, 8251, 2488, 382, 2212, 0])

'tiktoken is great!'

警告：虽然 `.decode()` 可以作用于单个 token，但要注意，对于不在 utf-8 边界上的 token，它可能会丢失信息。

对于单个 token，`.decode_single_token_bytes()` 可以安全地将单个整数 token 转换为它所代表的字节。

In [9]:
[encoding.decode_single_token_bytes(token) for token in [83, 8251, 2488, 382, 2212, 0]]


[b't', b'ikt', b'oken', b' is', b' great', b'!']

（字符串前面的 `b` 表示这些字符串是字节串。）

## 5. 比较不同编码

不同编码在拆分单词、组合空格以及处理非英文字符方面各不相同。利用上述方法，我们可以在几个示例字符串上比较不同的编码。

In [10]:
def compare_encodings(example_string: str) -> None:
    """Prints a comparison of three string encodings."""
    # print the example string
    print(f'\nExample string: "{example_string}"')
    # for each encoding, print the # of tokens, the token integers, and the token bytes
    for encoding_name in ["r50k_base", "p50k_base", "cl100k_base", "o200k_base"]:
        encoding = tiktoken.get_encoding(encoding_name)
        token_integers = encoding.encode(example_string)
        num_tokens = len(token_integers)
        token_bytes = [encoding.decode_single_token_bytes(token) for token in token_integers]
        print()
        print(f"{encoding_name}: {num_tokens} tokens")
        print(f"token integers: {token_integers}")
        print(f"token bytes: {token_bytes}")

In [11]:
compare_encodings("antidisestablishmentarianism")


Example string: "antidisestablishmentarianism"

r50k_base: 5 tokens
token integers: [415, 29207, 44390, 3699, 1042]
token bytes: [b'ant', b'idis', b'establishment', b'arian', b'ism']

p50k_base: 5 tokens
token integers: [415, 29207, 44390, 3699, 1042]
token bytes: [b'ant', b'idis', b'establishment', b'arian', b'ism']

cl100k_base: 6 tokens
token integers: [519, 85342, 34500, 479, 8997, 2191]
token bytes: [b'ant', b'idis', b'establish', b'ment', b'arian', b'ism']

o200k_base: 6 tokens
token integers: [493, 129901, 376, 160388, 21203, 2367]
token bytes: [b'ant', b'idis', b'est', b'ablishment', b'arian', b'ism']


In [12]:
compare_encodings("2 + 2 = 4")


Example string: "2 + 2 = 4"

r50k_base: 5 tokens
token integers: [17, 1343, 362, 796, 604]
token bytes: [b'2', b' +', b' 2', b' =', b' 4']

p50k_base: 5 tokens
token integers: [17, 1343, 362, 796, 604]
token bytes: [b'2', b' +', b' 2', b' =', b' 4']

cl100k_base: 7 tokens
token integers: [17, 489, 220, 17, 284, 220, 19]
token bytes: [b'2', b' +', b' ', b'2', b' =', b' ', b'4']

o200k_base: 7 tokens
token integers: [17, 659, 220, 17, 314, 220, 19]
token bytes: [b'2', b' +', b' ', b'2', b' =', b' ', b'4']


In [13]:
compare_encodings("お誕生日おめでとう")


Example string: "お誕生日おめでとう"

r50k_base: 14 tokens
token integers: [2515, 232, 45739, 243, 37955, 33768, 98, 2515, 232, 1792, 223, 30640, 30201, 29557]
token bytes: [b'\xe3\x81', b'\x8a', b'\xe8\xaa', b'\x95', b'\xe7\x94\x9f', b'\xe6\x97', b'\xa5', b'\xe3\x81', b'\x8a', b'\xe3\x82', b'\x81', b'\xe3\x81\xa7', b'\xe3\x81\xa8', b'\xe3\x81\x86']

p50k_base: 14 tokens
token integers: [2515, 232, 45739, 243, 37955, 33768, 98, 2515, 232, 1792, 223, 30640, 30201, 29557]
token bytes: [b'\xe3\x81', b'\x8a', b'\xe8\xaa', b'\x95', b'\xe7\x94\x9f', b'\xe6\x97', b'\xa5', b'\xe3\x81', b'\x8a', b'\xe3\x82', b'\x81', b'\xe3\x81\xa7', b'\xe3\x81\xa8', b'\xe3\x81\x86']

cl100k_base: 9 tokens
token integers: [33334, 45918, 243, 21990, 9080, 33334, 62004, 16556, 78699]
token bytes: [b'\xe3\x81\x8a', b'\xe8\xaa', b'\x95', b'\xe7\x94\x9f', b'\xe6\x97\xa5', b'\xe3\x81\x8a', b'\xe3\x82\x81', b'\xe3\x81\xa7', b'\xe3\x81\xa8\xe3\x81\x86']

o200k_base: 8 tokens
token integers: [8930, 9697, 243, 128225, 8930, 1769

## 6. 计算 Chat Completions API 调用的 token 数

像 `gpt-4o-mini` 和 `gpt-4` 这样的 ChatGPT 模型使用 token 的方式与较早的补全模型相同，但由于它们采用基于消息的格式，计算一段对话会用掉多少 token 更为困难。

下面是一个示例函数，用于计算传给 `gpt-3.5-turbo`、`gpt-4`、`gpt-4o` 和 `gpt-4o-mini` 的消息的 token 数。

注意，从消息中计算 token 的具体方式可能因模型而异。请将下面函数的计算结果视为估计值，而不是永久有效的保证。

特别是，使用可选 functions 输入的请求会在下面计算的估计值之外额外消耗 token。

In [14]:
def num_tokens_from_messages(messages, model="gpt-4o-mini-2024-07-18"):
    """Return the number of tokens used by a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        print("Warning: model not found. Using o200k_base encoding.")
        encoding = tiktoken.get_encoding("o200k_base")
    if model in {
        "gpt-3.5-turbo-0125",
        "gpt-4-0314",
        "gpt-4-32k-0314",
        "gpt-4-0613",
        "gpt-4-32k-0613",
        "gpt-4o-mini-2024-07-18",
        "gpt-4o-2024-08-06"
        }:
        tokens_per_message = 3
        tokens_per_name = 1
    elif "gpt-3.5-turbo" in model:
        print("Warning: gpt-3.5-turbo may update over time. Returning num tokens assuming gpt-3.5-turbo-0125.")
        return num_tokens_from_messages(messages, model="gpt-3.5-turbo-0125")
    elif "gpt-4o-mini" in model:
        print("Warning: gpt-4o-mini may update over time. Returning num tokens assuming gpt-4o-mini-2024-07-18.")
        return num_tokens_from_messages(messages, model="gpt-4o-mini-2024-07-18")
    elif "gpt-4o" in model:
        print("Warning: gpt-4o and gpt-4o-mini may update over time. Returning num tokens assuming gpt-4o-2024-08-06.")
        return num_tokens_from_messages(messages, model="gpt-4o-2024-08-06")
    elif "gpt-4" in model:
        print("Warning: gpt-4 may update over time. Returning num tokens assuming gpt-4-0613.")
        return num_tokens_from_messages(messages, model="gpt-4-0613")
    else:
        raise NotImplementedError(
            f"""num_tokens_from_messages() is not implemented for model {model}."""
        )
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>
    return num_tokens


In [15]:
# let's verify the function above matches the OpenAI API response

from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

example_messages = [
    {
        "role": "system",
        "content": "You are a helpful, pattern-following assistant that translates corporate jargon into plain English.",
    },
    {
        "role": "system",
        "name": "example_user",
        "content": "New synergies will help drive top-line growth.",
    },
    {
        "role": "system",
        "name": "example_assistant",
        "content": "Things working well together will increase revenue.",
    },
    {
        "role": "system",
        "name": "example_user",
        "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage.",
    },
    {
        "role": "system",
        "name": "example_assistant",
        "content": "Let's talk later when we're less busy about how to do better.",
    },
    {
        "role": "user",
        "content": "This late pivot means we don't have time to boil the ocean for the client deliverable.",
    },
]

for model in [
    "gpt-3.5-turbo",
    "gpt-4-0613",
    "gpt-4",
    "gpt-4o",
    "gpt-4o-mini"
    ]:
    print(model)
    # example token count from the function defined above
    print(f"{num_tokens_from_messages(example_messages, model)} prompt tokens counted by num_tokens_from_messages().")
    # example token count from the OpenAI API
    response = client.chat.completions.create(model=model,
    messages=example_messages,
    temperature=0,
    max_tokens=1)
    print(f'{response.usage.prompt_tokens} prompt tokens counted by the OpenAI API.')
    print()


gpt-3.5-turbo
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-4-0613
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-4
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-4o
124 prompt tokens counted by num_tokens_from_messages().
124 prompt tokens counted by the OpenAI API.

gpt-4o-mini
124 prompt tokens counted by num_tokens_from_messages().
124 prompt tokens counted by the OpenAI API.



## 7. 计算带工具调用的 chat completions 的 token 数

接下来，我们来看如何将这些计算应用于可能包含函数调用的消息。由于工具本身的格式，这并不是一件简单的事。 

下面是一个示例函数，用于计算传给 `gpt-3.5-turbo`、`gpt-4`、`gpt-4o` 和 `gpt-4o-mini` 的、包含工具的消息的 token 数。

In [16]:
def num_tokens_for_tools(functions, messages, model):
    
    # Initialize function settings to 0
    func_init = 0
    prop_init = 0
    prop_key = 0
    enum_init = 0
    enum_item = 0
    func_end = 0
    
    if model in [
        "gpt-4o",
        "gpt-4o-mini"
    ]:
        
        # Set function settings for the above models
        func_init = 7
        prop_init = 3
        prop_key = 3
        enum_init = -3
        enum_item = 3
        func_end = 12
    elif model in [
        "gpt-3.5-turbo",
        "gpt-4"
    ]:
        # Set function settings for the above models
        func_init = 10
        prop_init = 3
        prop_key = 3
        enum_init = -3
        enum_item = 3
        func_end = 12
    else:
        raise NotImplementedError(
            f"""num_tokens_for_tools() is not implemented for model {model}."""
        )
    
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        print("Warning: model not found. Using o200k_base encoding.")
        encoding = tiktoken.get_encoding("o200k_base")
    
    func_token_count = 0
    if len(functions) > 0:
        for f in functions:
            func_token_count += func_init  # Add tokens for start of each function
            function = f["function"]
            f_name = function["name"]
            f_desc = function["description"]
            if f_desc.endswith("."):
                f_desc = f_desc[:-1]
            line = f_name + ":" + f_desc
            func_token_count += len(encoding.encode(line))  # Add tokens for set name and description
            if len(function["parameters"]["properties"]) > 0:
                func_token_count += prop_init  # Add tokens for start of each property
                for key in list(function["parameters"]["properties"].keys()):
                    func_token_count += prop_key  # Add tokens for each set property
                    p_name = key
                    p_type = function["parameters"]["properties"][key]["type"]
                    p_desc = function["parameters"]["properties"][key]["description"]
                    if "enum" in function["parameters"]["properties"][key].keys():
                        func_token_count += enum_init  # Add tokens if property has enum list
                        for item in function["parameters"]["properties"][key]["enum"]:
                            func_token_count += enum_item
                            func_token_count += len(encoding.encode(item))
                    if p_desc.endswith("."):
                        p_desc = p_desc[:-1]
                    line = f"{p_name}:{p_type}:{p_desc}"
                    func_token_count += len(encoding.encode(line))
        func_token_count += func_end
        
    messages_token_count = num_tokens_from_messages(messages, model)
    total_tokens = messages_token_count + func_token_count
    
    return total_tokens

In [17]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_current_weather",
      "description": "Get the current weather in a given location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "The city and state, e.g. San Francisco, CA",
          },
          "unit": {"type": "string", 
                   "description": "The unit of temperature to return",
                   "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["location"],
      },
    }
  }
]

example_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that can answer to questions about the weather.",
    },
    {
        "role": "user",
        "content": "What's the weather like in San Francisco?",
    },
]

for model in [
    "gpt-3.5-turbo",
    "gpt-4",
    "gpt-4o",
    "gpt-4o-mini"
    ]:
    print(model)
    # example token count from the function defined above
    print(f"{num_tokens_for_tools(tools, example_messages, model)} prompt tokens counted by num_tokens_for_tools().")
    # example token count from the OpenAI API
    response = client.chat.completions.create(model=model,
          messages=example_messages,
          tools=tools,
          temperature=0)
    print(f'{response.usage.prompt_tokens} prompt tokens counted by the OpenAI API.')
    print()

gpt-3.5-turbo
105 prompt tokens counted by num_tokens_for_tools().
105 prompt tokens counted by the OpenAI API.

gpt-4
105 prompt tokens counted by num_tokens_for_tools().
105 prompt tokens counted by the OpenAI API.

gpt-4o
101 prompt tokens counted by num_tokens_for_tools().
101 prompt tokens counted by the OpenAI API.

gpt-4o-mini
101 prompt tokens counted by num_tokens_for_tools().
101 prompt tokens counted by the OpenAI API.

